# 01 — Data Exploration: Amharic Sentiment Dataset

**Project:** Developing Deep Learning-Based Sentiment Analysis of Amharic Social Media for Public Policy Enhancement in Ethiopia  
**Author:** Getnet Bogale  
**Datasets:**
- AfriSenti AMH (`data/raw/amh/`) — train / dev / test splits
- Policy-related tweets (`data/raw/dataset.xlsx`) — additionally collected Amharic tweets

**Goals of this notebook:**
1. Load and inspect both datasets
2. Analyse class (sentiment) distributions
3. Explore tweet length statistics
4. Visualise frequent tokens after preprocessing
5. Check for data quality issues (duplicates, missing values)

In [ ]:
import sys
sys.path.insert(0, '..')  # allow importing from src/

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from pathlib import Path
from collections import Counter

# Use a font that supports Ethiopic script if available
matplotlib.rcParams['font.family'] = ['DejaVu Sans', 'sans-serif']
sns.set_theme(style='whitegrid', palette='Set2')

DATA_RAW = Path('../data/raw')

## 1 · Load AfriSenti AMH splits

In [ ]:
def load_afrisenti_split(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, sep='\t', header=None, names=['tweet', 'label'])
    return df

train_df = load_afrisenti_split(DATA_RAW / 'amh/train.tsv')
dev_df   = load_afrisenti_split(DATA_RAW / 'amh/dev.tsv')
test_df  = load_afrisenti_split(DATA_RAW / 'amh/test.tsv')

print(f'Train: {len(train_df):,}  Dev: {len(dev_df):,}  Test: {len(test_df):,}')
train_df.head()

## 2 · Class distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (df, split) in zip(axes, [(train_df, 'Train'), (dev_df, 'Dev'), (test_df, 'Test')]):
    counts = df['label'].value_counts()
    counts.plot.bar(ax=ax, color=['#4CAF50', '#F44336', '#2196F3'])
    ax.set_title(f'{split} ({len(df):,} tweets)')
    ax.set_xlabel('Sentiment')
    ax.set_ylabel('Count')
    ax.tick_params(axis='x', rotation=0)
    for bar in ax.patches:
        ax.annotate(f'{int(bar.get_height()):,}',
                    (bar.get_x() + bar.get_width() / 2, bar.get_height()),
                    ha='center', va='bottom', fontsize=10)
plt.suptitle('AfriSenti AMH — Sentiment Class Distribution', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('../results/figures/class_distribution_afrisenti.png', dpi=150, bbox_inches='tight')
plt.show()

## 3 · Load additionally collected policy tweets

In [ ]:
policy_df = pd.read_excel(DATA_RAW / 'dataset.xlsx')
print(f'Policy tweets: {len(policy_df):,}')
print('Columns:', policy_df.columns.tolist())
policy_df.head()

## 4 · Tweet length analysis

In [ ]:
train_df['char_len'] = train_df['tweet'].str.len()
train_df['word_len'] = train_df['tweet'].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
train_df['char_len'].plot.hist(bins=50, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Character length distribution (train)')
axes[0].set_xlabel('Characters')

train_df['word_len'].plot.hist(bins=40, ax=axes[1], color='coral', edgecolor='white')
axes[1].set_title('Word count distribution (train)')
axes[1].set_xlabel('Words')

plt.tight_layout()
plt.savefig('../results/figures/tweet_length_distribution.png', dpi=150)
plt.show()

print(train_df[['char_len', 'word_len']].describe())

## 5 · Preprocessing & token frequency

In [ ]:
from src.preprocessing.preprocess import preprocess_text

# Apply pipeline (character normalisation + noise removal + stopword removal)
train_df['tokens'] = train_df['tweet'].apply(preprocess_text)

all_tokens = [tok for toks in train_df['tokens'] for tok in toks]
freq = Counter(all_tokens)
print(f'Vocabulary size: {len(freq):,}')
print('Top-20 tokens:', freq.most_common(20))

In [ ]:
top_n = 30
top_tokens, top_counts = zip(*freq.most_common(top_n))

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(range(top_n), top_counts, color='mediumpurple')
ax.set_xticks(range(top_n))
ax.set_xticklabels(top_tokens, rotation=45, ha='right', fontsize=9)
ax.set_title(f'Top {top_n} most frequent Amharic tokens (after stopword removal)')
ax.set_ylabel('Frequency')
plt.tight_layout()
plt.savefig('../results/figures/top_tokens.png', dpi=150)
plt.show()

## 6 · Data quality checks

In [ ]:
print('=== Missing values ===')
print(train_df.isnull().sum())

print('\n=== Duplicate tweets ===')
dups = train_df.duplicated(subset=['tweet'])
print(f'  Exact duplicates: {dups.sum():,}')

print('\n=== Empty after preprocessing ===')
empty = train_df['tokens'].map(len) == 0
print(f'  Empty token lists: {empty.sum():,}')

## 7 · Summary table

In [ ]:
summary = pd.DataFrame({
    'Split':    ['Train', 'Dev', 'Test', 'Policy (xlsx)'],
    'Tweets':   [len(train_df), len(dev_df), len(test_df), len(policy_df)],
    'Source':   ['AfriSenti', 'AfriSenti', 'AfriSenti', 'Collected'],
})
display(summary)

print('\nSentiment counts (train):')
display(train_df['label'].value_counts().to_frame('count'))